# Sesión 4 — Taller práctico: del hallazgo a la decisión (CONECTA, proyecto A: Retención)

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 4 de 9  
**Tipo:** Notebook práctico

## Instrucciones de entrega

- Equipos de 3 personas. Entreguen este notebook **ejecutado de inicio a fin** como
  `DATAVIZ_sesion_4_proyecto_A_grupo_Y.ipynb` en el plazo acordado en clase.
- Completen las celdas `# TODO` y escriban en las celdas **Su respuesta**.
- Pueden consultar el notebook guiado, pero escriban su propio código. Pueden apoyarse en un asistente de IA para
  concentrarse en lo importante y en las decisiones; declaren al final para qué lo usaron.

## Problemática de negocio

CONECTA S.A. es un operador de telecomunicaciones en Bogotá con dos líneas, **Hogar** (internet fijo) y **Móvil**
(pospago y prepago), con datos sintéticos del curso. En diciembre, el comité directivo reparte el presupuesto del
primer trimestre de 2027 entre las propuestas de las gerencias. La **Gerencia de Retención** quiere una campaña para
reducir las cancelaciones. Como analistas, deben:

1. Partir del hallazgo candidato de la S3: las diez UPZ con más pérdida (el 10 % peor entre las UPZ con al menos 80
   clientes).
2. Verificar si resiste la pregunta del comité: «¿no será que tienen más clientes móviles prepago?».
3. Encontrar en qué segmento está el problema y qué lo explica (y qué no).
4. Decir cuánto vale actuar, en pesos y con un escenario declarado.
5. Construir el gráfico principal para el comité y recomendar dónde y para quién hacer la campaña, y cuánto puede
   costar como máximo.

La Tarea 3 es propia de CONECTA (Golden State no tiene segmentos): la presentación la explica en el bloque del taller.
La carpeta `datos/` no se modifica.

## Preparación del entorno

In [ ]:
import pathlib
import textwrap

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

In [ ]:
FUENTE_CONECTA = "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026). Cálculos del curso."
# Tabla limpia que dejó la S3: el «cambio de plan» ya no cuenta como pérdida (columna perdida).
d = pd.read_csv(DATOS / "conecta_limpio_s3.csv", dtype={"mes": str})
d["segmento"] = np.where(d["linea"] == "Hogar", "Hogar", np.where(d["modalidad"] == "Prepago", "Móvil prepago", "Móvil pospago"))
upz = gpd.read_file(DATOS / "upz.geojson")
MIN_CLIENTES, PEOR, UMBRAL, ESCENARIOS = 80, 0.10, 1.5, (0.25, 0.50)
print(f"{miles(d['cliente_id'].nunique())} clientes; {miles(len(d))} filas cliente-mes; pérdida mensual de Bogotá: "
      f"{decimal(100 * d['perdida'].mean(), 2)} %")
display((100 * d.groupby("segmento")["perdida"].mean()).round(2).to_frame("pérdida mensual (%)"))

## Tarea 1 — Del hallazgo candidato a la idea central

La S3 (Tarea 5) entregó **diez UPZ candidatas**: entre las UPZ con al menos `MIN_CLIENTES` clientes, el 10 % con la
tasa de pérdida más alta. La celda siguiente **ya está hecha**: rehace esa tabla con la misma regla, porque la
verificación necesita sus columnas (`clientes`, `cliente_mes`, `perdidas`, `pct_prepago`, `tasa_pct`, `candidata`).
En esta tarea solo se lee la tabla para escribir la primera idea central.

In [ ]:
# Ya está hecha: las candidatas de la S3 (la regla de la S3, Tarea 5).
por_upz = d.groupby("codigo_upz").agg(upz=("upz", "first"), localidad=("localidad", "first"),
                                    clientes=("cliente_id", "nunique"), cliente_mes=("perdida", "size"),
                                    perdidas=("perdida", "sum"),
                                    pct_prepago=("modalidad", lambda s: 100 * (s == "Prepago").mean()))
por_upz["tasa_pct"] = 100 * por_upz["perdidas"] / por_upz["cliente_mes"]
por_upz = por_upz[por_upz["clientes"] >= MIN_CLIENTES].copy()
k = round(PEOR * len(por_upz))
por_upz["candidata"] = por_upz.index.isin(por_upz.nlargest(k, "tasa_pct").index)
display(por_upz[por_upz["candidata"]].sort_values("tasa_pct", ascending=False).round(2))
veces = por_upz.loc[por_upz["candidata"], "tasa_pct"] / (100 * d["perdida"].mean())
print(f"{k} candidatas; pierden entre {decimal(veces.min(), 1)} y {decimal(veces.max(), 1)} veces la tasa de Bogotá")

**Escriban la primera idea central en una oración (para quién, qué decisión y qué hallazgo) y la pregunta que haría el comité al ver la columna pct_prepago.**

*Consulten en la presentación:* diapositiva 10, «Análisis exploratorio frente a explicativo» (sección «Del hallazgo a la idea central»); diapositiva 12, «Quién, qué y cómo; la idea central» (sección «Del hallazgo a la idea central»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 2 — Verificar contra comparables

Los comparables de cada cliente son los de **su mismo segmento** (Hogar, Móvil pospago o Móvil prepago) en toda
Bogotá: la tabla de la preparación muestra que el móvil prepago pierde mucho más que los otros dos. Las **pérdidas
esperadas** de una UPZ son las que tendría si cada cliente perdiera como su segmento en Bogotá; si la UPZ pierde
1,5 veces lo esperado o más, el exceso no se explica por su mezcla de segmentos. La presentación lo resuelve a mano
con una UPZ que no es candidata.

1. Calculen, fila por fila, la columna `esperada` (la tasa de pérdida de su segmento).
2. Por UPZ: `esperadas = suma de esperada`, `razon = perdidas / esperadas` y `exceso = perdidas - esperadas`.
3. Una candidata se **confirma** si `razon >= UMBRAL`. Repitan con 1,3 y 1,7 (robustez: como el 4 % y el 6 % de
   Golden State).
4. Hagan el gráfico de respaldo: pérdida observada frente a esperada en las diez candidatas.

In [ ]:
# TODO: calculen esperada por fila, las columnas por UPZ y la columna confirmada.
# d["esperada"] = d.groupby("segmento")["perdida"].transform("mean")
# por_upz["esperadas"] = ...; por_upz["razon"] = ...; por_upz["exceso"] = ...
# por_upz["confirmada"] = ...
# Su código aquí:

In [ ]:
# TODO: hagan el gráfico de respaldo (observada frente a esperada) con un título que diga la conclusión.
# fig, ax = plt.subplots(figsize=(9, 4.6))
# Su código aquí:

**¿Cuántas candidatas se confirman, cuáles se descartan y por qué? ¿Cambia con el umbral?**

*Consulten en la presentación:* diapositiva 49, «Cómo se verifica en CONECTA» (sección «El taller: CONECTA»); diapositiva 48, «Los segmentos de clientes» (sección «El taller: CONECTA»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 3 — ¿En qué segmento está el problema y qué lo explica?

Antes de valorar la campaña hay que saber **a quién** va dirigida y **por qué** se van esos clientes.

1. **Dónde:** en las UPZ confirmadas y en el resto de Bogotá, calculen la razón observada/esperada **por segmento**
   (`razon_segmento`).
2. **Qué lo acompaña:** solo para los clientes de **Hogar**, comparen las confirmadas con el resto en quejas por mes,
   uso de datos (mediana), lo que pagan, porcentaje de pérdidas con menos de seis meses de antigüedad (la medida
   de la S2), NPS medio, porcentaje de pérdidas voluntarias y tasa de pérdida
   antes y desde agosto de 2026 (`mecanismo`).

In [ ]:
# TODO: calculen razon_segmento y la tabla mecanismo (solo Hogar).
# conf = por_upz[por_upz["confirmada"]]
# grupo = d["codigo_upz"].isin(conf.index).map({True: "UPZ confirmadas", False: "Resto de Bogotá"})
# t = d.groupby([grupo, "segmento"]).agg(perdidas=("perdida", "sum"), esperadas=("esperada", "sum"))
# razon_segmento = (t["perdidas"] / t["esperadas"]).unstack()
# h = d[d["segmento"] == "Hogar"]; gh = grupo.loc[h.index]
# mecanismo = pd.DataFrame({"quejas_mes": h.groupby(gh)["quejas_mes"].mean(), ...}).T
# Su código aquí:

**¿En qué segmento está el exceso? ¿Qué lo explica y qué explicaciones quedan descartadas?**

*Consulten en la presentación:* diapositiva 50, «Tarea 3 en CONECTA» (sección «El taller: CONECTA»); diapositiva 31, «Lista de verificación previa al comité» (sección «Cuánto vale y el SCQA»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 4 — Cuánto vale y el SCQA

La campaña va para los clientes de **Hogar** de las UPZ confirmadas (Tarea 3).

1. **Magnitud:** exceso de pérdidas de Hogar en el año (`exceso` = pérdidas − esperadas, solo filas de Hogar de las
   confirmadas), ARPU de esos clientes, ingreso anual que se va (`exceso × ARPU × 12`) y ganancia de la campaña en
   cada escenario (`× fracción recuperada`).
   - **ARPU** (ingreso medio por cliente y mes, definición del curso): el promedio de `ingreso_mensual_cop` en las
     filas cliente-mes de Hogar de las UPZ confirmadas. Se promedia por fila: cada mes de cada cliente pesa lo mismo.
   - **Fracción recuperada:** una cuarta parte (escenario prudente) y la mitad. Es un supuesto del caso, no un dato:
     los datos no dicen cuántos clientes recuperaría la campaña. Por eso se muestran dos escenarios.
2. **Costo máximo por cliente:** ganancia en el escenario prudente / clientes activos. **Clientes activos:** los
   `cliente_id` de Hogar distintos que tienen fila en el último mes de la tabla (`d["mes"].max()`, octubre de 2026)
   en las UPZ confirmadas.
3. **Estabilidad (robustez por periodo):** tasa mensual de los clientes de Hogar de las confirmadas frente a los del
   resto de Bogotá (gráfico de líneas). Es la evidencia para la robustez por periodo de la lista de verificación.

*Consulten en la presentación:* diapositiva 51, «El mismo razonamiento en CONECTA» (sección «El taller: CONECTA»); diapositiva 22, «Cuánto vale actuar» (sección «Cuánto vale y el SCQA»).

In [ ]:
# TODO: calculen la magnitud, la ganancia por escenario y el costo máximo por cliente.
# en_conf = h[h["codigo_upz"].isin(conf.index)]          # filas de Hogar de las confirmadas
# exceso = en_conf["perdida"].sum() - en_conf["esperada"].sum(); arpu = ...
# ganancia = {f: ... for f in ESCENARIOS}; activos = ...; costo_maximo = ...
# Su código aquí:

In [ ]:
# TODO: hagan la serie mensual de los clientes de Hogar: confirmadas frente al resto de Bogotá.
# serie = 100 * h.groupby(["mes", gh])["perdida"].mean().unstack()     # h y gh vienen de la Tarea 3
# Su código aquí:

**Escriban el SCQA para el comité (S, C, Q y A). Partan de lo que traemos de la S1 a la S3: ¿qué hallazgo se confirmó, cuál resultó ser la mezcla de segmentos y cuál no explica las zonas?**

*Consulten en la presentación:* diapositiva 28, «SCQA: el orden que necesita la audiencia» (sección «Cuánto vale y el SCQA»); diapositiva 46, «LO QUE TRAEMOS DE CONECTA» (sección «El taller: CONECTA»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**Lista de verificación: magnitud, mecanismo, robustez (umbral y periodo: usen la serie mensual), explicaciones alternativas y accionabilidad.**

*Consulten en la presentación:* diapositiva 31, «Lista de verificación previa al comité» (sección «Cuánto vale y el SCQA»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 5 — El gráfico principal y la recomendación

Primero miren el gráfico «antes»: la cancelación por localidad que habría llevado al comité un equipo que se saltara
la S3 (la celda ya está hecha). Luego construyan el gráfico de decisión: a la izquierda, el mapa de las UPZ
confirmadas, y a la derecha, un panel con la ganancia anual por localidad (escenario prudente), con un título que diga
la conclusión. **El mapa ya está hecho** (función `mapa_confirmadas`): los mapas son el tema de la S5, donde se
construyen y se discute a qué escala mirarlos. Ustedes hacen el panel, el título y la recomendación. La ganancia de cada localidad usa el ARPU de los clientes de Hogar de esa localidad (regla del curso),
así que la suma por localidad no da exactamente el total de la Tarea 4: hay que decirlo donde aparezcan las dos cifras.

**Lista de chequeo del gráfico principal** (la misma de la presentación; sirve para revisar el gráfico antes de entregarlo, no se responde por escrito):

1. ¿El título dice la conclusión y la decisión que debe tomar el comité?
2. ¿Las barras empiezan en cero?
3. ¿Un color para el mensaje y gris para el contexto?
4. ¿Etiquetas directas en lugar de ejes y leyendas?
5. ¿El escenario y la fuente están declarados?
6. ¿El mapa muestra el *dónde* y las barras el *cuánto*?

*Consulten en la presentación:* diapositiva 35, «Gráfico de decisión frente a gráfico de evidencia» (sección «El gráfico principal»); diapositiva 40, «Las seis decisiones de diseño» (sección «El gráfico principal»); diapositiva 31, «Lista de verificación previa al comité» (sección «Cuánto vale y el SCQA»).

In [ ]:
# Gráfico «antes» (ya está hecho): los datos crudos, la columna cancelo tal cual y el estilo por defecto.
crudo = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str})
antes_loc = crudo.groupby("localidad")["cancelo"].mean()
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(12, 4.5))
    antes_loc.plot.bar(ax=ax)
    ax.set_title("cancelo")
    ax.set_ylabel("cancelo")
    plt.show()
print(f"{antes_loc.size} barras; la tabla limpia de la S3 tiene {d['localidad'].nunique()} localidades")

In [ ]:
# Ya está hecha (los mapas son el tema de la S5): dibuja las UPZ con las confirmadas resaltadas y sus nombres.
def mapa_confirmadas(ax, confirmadas):
    es_conf = upz["codigo_upz"].isin(confirmadas.index)
    upz[~es_conf].plot(ax=ax, color=PALETA["fondo"], edgecolor="white", linewidth=0.5)
    upz[es_conf].plot(ax=ax, color=PALETA["principal"], edgecolor="white", linewidth=0.8)
    # Rótulos en una columna a la izquierda, ordenados por latitud y separados para que no se monten.
    puntos = sorted(((f.geometry.representative_point(), f["upz"].title()) for _, f in upz[es_conf].iterrows()),
                    key=lambda x: x[0].y)
    x_rotulo, y_previo = upz.total_bounds[0] + 0.02, -1e9
    for punto, nombre in puntos:
        y_rotulo = max(punto.y, y_previo + 0.018); y_previo = y_rotulo
        ax.annotate(nombre, (punto.x, punto.y), xytext=(x_rotulo, y_rotulo), textcoords="data", fontsize=9.5, ha="right",
                    va="center", arrowprops={"arrowstyle": "-", "color": PALETA["secundario"], "linewidth": 0.6})
    ax.set_axis_off()

In [ ]:
# TODO: calculen la ganancia por localidad y hagan el panel, el título y la composición con el mapa.
# arpu_loc = en_conf.groupby("localidad")["ingreso_mensual_cop"].mean()
# por_localidad = en_conf.groupby("localidad").agg(perdidas=("perdida", "sum"), esperadas=("esperada", "sum"))
# fig = plt.figure(figsize=(11, 7.2))
# mapa_confirmadas(fig.add_axes([0.0, 0.0, 0.5, 0.93]), conf)      # el mapa ya está hecho
# px = fig.add_axes([0.55, 0.05, 0.43, 0.86])                      # el panel de barras es suyo
# Su código aquí:

**Diagnostiquen el gráfico «antes» (cancelación cruda por localidad): ¿es feo, malo o erróneo? ¿Por qué?**

*Consulten en la presentación:* diapositiva 36, «Antes del gráfico principal» (sección «El gráfico principal»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**Escriban la recomendación con su magnitud y lo que el comité preguntará después.**

*Consulten en la presentación:* diapositiva 31, «Lista de verificación previa al comité» (sección «Cuánto vale y el SCQA»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Preguntas de interpretación

**1. Con la columna cancelo (sin la limpieza de la S3), ¿qué habría pasado con la recomendación?**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**2. ¿Qué evidencia hay de que el problema no es la red ni el precio? ¿Qué no podemos afirmar? (Tarea 3)**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

**3. El NPS de las confirmadas es más bajo, pero responde menos de una de cada cinco personas. ¿Qué tan confiable es?**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Interpretación final para el negocio

Para el comité directivo, en 5–8 viñetas: hallazgo verificado y para quién; magnitud en pesos con el escenario
declarado; recomendación y costo máximo; riesgos, límites de los datos y datos sensibles; qué información adicional
se necesitaría; uso de IA.

**Su respuesta:** *(Escriba aquí sus viñetas.)*